# 显存去哪了：tensor 与 nvidia-smi

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-04-where-did-vram-go.ipynb)

动手观察显存。边在 GPU 上分配 tensor、边盯 nvidia-smi，看清 PyTorch 的 caching allocator 与 nvidia-smi 的关系，这是理解 HAMi 显存隔离（gpumem）的物理基础。

本 notebook 配合教程章节 [显存去哪了：tensor 与 nvidia-smi](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

[Lab 01](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-01-meet-your-gpu/) 里我们看到 `Memory-Usage: 0MiB / 15360MiB`。当你开始分配 tensor，显存会涨，但涨的方式可能出乎意料。理解 **“PyTorch 已经认为释放了、但 `nvidia-smi` 还显示占用”** 这一现象，是理解 HAMi 的 `nvidia.com/gpumem`（显存配额隔离）为什么必要、以及它实际在管什么的物理基础。

## 第 1 步：基线

In [ ]:
import torch, subprocess

def gpu_mem_mib():
    out = subprocess.check_output(["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"]).decode().strip()
    return int(out.splitlines()[0])

print("before any CUDA call :", gpu_mem_mib(), "MiB")
torch.cuda.init(); torch.zeros(1, device="cuda")   # 强制创建 CUDA context
print("after context init   :", gpu_mem_mib(), "MiB")

预期：第一行是 **0 MiB**（`import torch` 不会碰 GPU）；第二行跳到几百 MiB。这几百 MiB 是 **CUDA context**：驱动为这个进程在显存里建的页表、kernel 镜像、cuBLAS/cuDNN 工作区等。它不属于任何 tensor，也不会被 `empty_cache()` 释放，**每个用 GPU 的进程都要交这笔“入场费”**。

> **context 开销为什么对共享很重要**
>
> 如果 10 个 Pod 共享一张 T4，光 context 就要吃掉几 GB。[Lab 09](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/) 的 MPS 让多个进程共用一个服务端 context，部分动机就是省下这笔费用。

## 第 2 步：分配一个大 tensor

In [ ]:
# 2 GiB 的 float32（2*1024*1024*1024 字节）
a = torch.randn(2 * 1024 * 1024 * 1024 // 4, device="cuda")
print("after alloc  nvidia-smi:", gpu_mem_mib(), "MiB")
print("torch sees allocated   :", round(torch.cuda.memory_allocated()/1024**2), "MiB")
print("torch sees reserved    :", round(torch.cuda.memory_reserved()/1024**2), "MiB")

预期：`nvidia-smi` 在 context 基线之上再涨约 2048 MiB；`memory_allocated` ≈ 2048；`memory_reserved` ≈ 2048（单个大块分配时 caching allocator 通常正好预留这么多；如果你分配的是许多小 tensor，`reserved` 会明显大于 `allocated`，因为它按 2 MiB / 20 MiB 的块向驱动要显存）。

## 第 3 步：删掉 tensor，显存为什么不立刻还？

In [ ]:
del a
print("after del  nvidia-smi:", gpu_mem_mib(), "MiB")
print("torch sees allocated :", round(torch.cuda.memory_allocated()/1024**2), "MiB")
print("torch sees reserved  :", round(torch.cuda.memory_reserved()/1024**2), "MiB")

你会看到：

- `torch.cuda.memory_allocated()` 立刻回到很低（PyTorch 认为 tensor 已释放）；
- 但 `nvidia-smi` 和 `memory_reserved()` 往往**没掉**：PyTorch 的 **caching allocator** 把这块显存攥在手里，准备下次复用，而不是立刻还给驱动。

![del 之后 tensor 显存进入 PyTorch caching allocator 的 free pool 等待复用，只有 empty_cache 才还给驱动](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/caching-allocator-pool.svg)

## 第 4 步：强制还给驱动

In [ ]:
torch.cuda.empty_cache()
print("after empty_cache  nvidia-smi:", gpu_mem_mib(), "MiB")
print("torch sees reserved        :", round(torch.cuda.memory_reserved()/1024**2), "MiB")

现在 `nvidia-smi` 应该回落到 第 1 步 的 context 基线附近。`empty_cache()` 只释放 caching allocator 的空闲块，**不影响**正在使用的 tensor，也**不会**释放 context。

> **为什么这件事很重要**
>
> 在真实多租户平台里，多个 Pod 共用一张 GPU。如果某个进程的 caching allocator 攥着 2 GB 不还，别的进程（Pod）就拿不到这 2 GB。**HAMi 的显存配额隔离（`nvidia.com/gpumem`）管的就是这一层：按 `nvidia-smi` 口径的“真实占用”给每个 Pod 设上限**，而不是相信每个进程都会自觉 `empty_cache`。这是你在 Colab 单机就能亲眼看到的、HAMi 要解决的物理问题。

## 验证

完成后你应该能解释：

- 为什么 `del a` 之后 `nvidia-smi` 的显存可能不掉；
- `memory_allocated` 与 `memory_reserved` 的区别；
- `empty_cache()` 做了什么、不做什么；
- 这和 HAMi 显存隔离有什么关系。

## 故障模式



| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| 分配时报 `CUDA out of memory` | 之前 session 残留 / 要得太接近 15 GB | 重启 runtime，或减小 tensor |
| `del` 后 `nvidia-smi` 没掉 | caching allocator 正常行为 | `torch.cuda.empty_cache()` |
| `empty_cache` 后显存仍有几百 MiB | CUDA context 常驻 | 正常，进程退出才会归零 |
| `del a` 后 `memory_allocated` 没降 | notebook 里还有变量引用它（如 `_` 或输出历史） | 避免在 cell 末尾裸写 `a`；必要时 `import gc; gc.collect()` |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。